# Project 2 · Experiment 2 — RoBERTa LOGO Evaluation
**Jupyter Notebook · Windows/Anaconda/NVIDIA · 与之前 IID Notebook 对齐**

目标：分别保留 GPT-4、ChatGPT、Llama 和 Mistral 作为**完全未见过的 AI 生成器**，从 `FacebookAI/roberta-base` **重新初始化**分类器并训练，只在固定的同一组测试文本上评估。**不得在 LOGO 中加载已经训练过全部四个生成器的 IID checkpoint。**

核心规则：
- IID 的 `iid_train.csv` / `iid_validation.csv` / `iid_test.csv` 不再随机划分；每个 LOGO fold 仅从 Train 和 Validation 删除一个目标 AI 生成器。Human 保留。
- 使用 IID 已保存的 `per_generator_test_manifest.csv`，与 IID 分生成器的结果进行**同一测试样本**比较；测试集不用于调参。
- 复用 IID 的主要训练参数：RoBERTa-base、256 tokens、3 epochs、batch size 4、grad accumulation 4、学习率 2e-5。每次训练都从新的预训练模型初始化。
- 每个 held-out 生成器都有单独的模型、输出目录和指标；避免覆盖 IID 结果。 
- 发布版本默认运行四个 fold；如需先检查流程，可把 `TARGETS_TO_RUN` 改为 `["llama-chat"]`。

**学术限制：** 训练时去掉一个生成器也减少了训练样本，IID–LOGO 差异不能单凭这一设计完全归因于“生成器未见过”。另外，不要使用 LOGO 测试成绩重新调训练参数。

## 0. 文件和环境

将本 Notebook 放在仓库的 `notebooks/` 文件夹，使用与 IID 实验**同一个 Jupyter Kernel**。运行 LOGO 前必须已生成 IID 结果文件；不需要重新训练 IID。项目路径会自动查找。

```text
raid-roberta-cross-generator-detection/
├── notebooks/
│   └── RAID_RoBERTa_LOGO_Jupyter.ipynb     ← 本文件
├── raid_experiment_data/
│   ├── iid_train.csv
│   ├── iid_validation.csv
│   └── iid_test.csv
├── iid_roberta_results_jupyter/
│   ├── iid_per_generator_metrics.csv
│   ├── iid_test_metrics.json
│   └── per_generator_test_manifest.csv
└── logo_roberta_results_jupyter/           ← 自动创建
```

本 Notebook 不修改 `raid_experiment_data` 和 `iid_roberta_results_jupyter` 中的任何文件。

In [ ]:
from pathlib import Path
import hashlib, json, math, random, gc, platform
import numpy as np
import pandas as pd
import torch
import transformers
from IPython.display import display
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

print("Python:", platform.python_version())
print("torch:", torch.__version__, "| transformers:", transformers.__version__)
print("CUDA usable:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("当前 Jupyter Kernel 看不到 CUDA。请切换回成功训练 IID 的同一个 Kernel。")
DEVICE = torch.device("cuda")
print("NVIDIA GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## 1. 运行配置

默认 `TARGETS_TO_RUN = ALL_GENERATORS`，依次完成四个 held-out 生成器实验。如需先运行一个 fold，可改为 `["llama-chat"]`；随后再改回 `ALL_GENERATORS` 并从头运行。完成且数据散列、超参数一致的 fold 会被跳过，不会重复训练。同一次研究的全部 fold 应保持超参数一致。


In [ ]:
def find_project_root() -> Path:
    """Find repository root from Jupyter launched in the root or notebooks/."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "raid_experiment_data").is_dir():
            return candidate
    for candidate in (start, *start.parents):
        if (candidate / "scripts").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "Cannot locate the repository root. Run Jupyter inside the cloned "
        "repository and generate raid_experiment_data/ using the scripts first."
    )

PROJECT_DIR = find_project_root()
DATA_DIR = PROJECT_DIR / "raid_experiment_data"
IID_RESULTS = PROJECT_DIR / "iid_roberta_results_jupyter"
LOGO_RESULTS = PROJECT_DIR / "logo_roberta_results_jupyter"

ALL_GENERATORS = ["chatgpt", "gpt4", "llama-chat", "mistral-chat"]
TARGETS_TO_RUN = ALL_GENERATORS  # 可先设为 ["llama-chat"] 进行单组检查

MODEL_NAME = "FacebookAI/roberta-base"
MAX_LENGTH = 256
EPOCHS = 3
BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
PATIENCE = 2
SEED = 42

assert TARGETS_TO_RUN and len(TARGETS_TO_RUN) == len(set(TARGETS_TO_RUN))
assert set(TARGETS_TO_RUN) <= set(ALL_GENERATORS)
assert DATA_DIR.is_dir(), f"找不到 {DATA_DIR}，请修改 PROJECT_DIR"
assert IID_RESULTS.is_dir(), f"找不到 {IID_RESULTS}，请核对文件夹名"
print("项目路径：", PROJECT_DIR.resolve())
print("本次执行的 held-out 生成器：", TARGETS_TO_RUN)
print("本次不会重新运行 IID。")

## 2. 读取 CSV，独立执行防泄漏审计（尚未开始训练）

这一格使用 IID 的三个固定划分，检查跨 split 是否存在相同 `source_id`、文本或 `id`，然后逐项验证 IID 测试清单与 `iid_test.csv` 的一致性。若失败，**不要跳过错误直接训练**。

In [ ]:
SPLIT_FILES = {"train": "iid_train.csv", "validation": "iid_validation.csv", "test": "iid_test.csv"}
REQUIRED = {"id", "source_id", "text", "label", "model", "domain"}
MANIFEST_PATH = IID_RESULTS / "per_generator_test_manifest.csv"
IID_PER_GEN_PATH = IID_RESULTS / "iid_per_generator_metrics.csv"
IID_METRICS_PATH = IID_RESULTS / "iid_test_metrics.json"

def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def clean_source_key(df):
    src = df["source_id"].astype(str).str.strip()
    return np.where(src.ne(""), "src::" + src, "missing::" + df["id"].astype(str))

def read_split(name, filename):
    p = DATA_DIR / filename
    if not p.is_file():
        raise FileNotFoundError(f"缺少必要文件：{p}")
    df = pd.read_csv(p, encoding="utf-8-sig", keep_default_na=False)
    missing = REQUIRED - set(df.columns)
    if missing: raise ValueError(f"{filename} 缺少字段: {sorted(missing)}")
    if df.empty or df["id"].duplicated().any():
        raise ValueError(f"{filename} 为空或者 id 重复")
    for col in ["id", "model", "domain", "text"]:
        if df[col].astype(str).str.strip().eq("").any():
            raise ValueError(f"{filename} 的 {col} 存在空值")
    df["label"] = pd.to_numeric(df["label"], errors="raise").astype(int)
    if set(df["label"].unique()) != {0, 1}:
        raise ValueError(f"{filename} 必须同时含有标签 0 与 1")
    if (df["model"].eq("human") != df["label"].eq(0)).any():
        raise ValueError(f"{filename} model 与 label 对应错误")
    df["_group"] = clean_source_key(df)
    df["_text_hash"] = df["text"].map(lambda t: hashlib.sha256(t.encode("utf-8")).hexdigest())
    return df

splits = {name: read_split(name, filename) for name,filename in SPLIT_FILES.items()}
for i,a in enumerate(SPLIT_FILES):
    for b in list(SPLIT_FILES)[i+1:]:
        left,right=splits[a],splits[b]
        overlaps={
            "source_group": len(set(left["_group"]) & set(right["_group"])),
            "exact_text": len(set(left["_text_hash"]) & set(right["_text_hash"])),
            "id": len(set(left["id"]) & set(right["id"]))
        }
        print(f"Leakage {a} vs {b}:", overlaps)
        if any(overlaps.values()): raise ValueError("跨 split 检测到泄漏，不能训练。")

manifest = pd.read_csv(MANIFEST_PATH, keep_default_na=False, encoding="utf-8-sig")
iid_by_gen = pd.read_csv(IID_PER_GEN_PATH, keep_default_na=False, encoding="utf-8-sig")
iid_main = json.loads(IID_METRICS_PATH.read_text(encoding="utf-8"))
assert set(manifest.columns) >= {"target_generator", "id", "source_id", "model", "domain", "label"}
assert set(iid_by_gen["target_generator"]) == set(ALL_GENERATORS)
assert set(manifest["target_generator"]) == set(ALL_GENERATORS)
assert not manifest.duplicated(["target_generator","id"]).any(), "目标生成器内 id 不能重复"
assert len(splits["test"]) == iid_main["test_data_count"], "IID Test 样本量不一致"
by_id=splits["test"].set_index("id", drop=False)
assert by_id.index.is_unique
for gen in ALL_GENERATORS:
    sub=manifest.loc[manifest["target_generator"] == gen].copy()
    assert set(sub["id"]) <= set(by_id.index), f"{gen}: manifest 包含不在 iid_test.csv 的 id"
    expected=by_id.loc[sub["id"].tolist()].reset_index(drop=True)
    for col in ["source_id", "model", "domain", "label"]:
        if not (sub[col].astype(str).reset_index(drop=True) == expected[col].astype(str)).all():
            raise ValueError(f"{gen}: manifest 的 {col} 与 iid_test.csv 不一致")
    assert set(expected.loc[expected["label"]==1,"model"]) == {gen}
    assert set(expected.loc[expected["label"]==0,"model"]) == {"human"}
    assert (expected["label"]==0).sum() == (expected["label"]==1).sum()
    assert expected.groupby("domain")["label"].sum().equals(expected.groupby("domain").size()//2), f"{gen}: domain 未配对平衡"
    ai_test_ids=set(splits["test"].loc[splits["test"]["model"] == gen, "id"])
    assert ai_test_ids == set(expected.loc[expected["label"]==1, "id"]), f"{gen}: AI Test 样本清单不完整"
    existing=iid_by_gen.loc[iid_by_gen["target_generator"] == gen].iloc[0]
    assert int(existing["n"]) == len(expected), f"{gen}: IID 指标数量与清单不一致"
    print(f"[CHECK OK] {gen}: {len(expected)} rows = {len(expected)//2} Human + {len(expected)//2} AI")

print("[PREFLIGHT PASSED] 固定划分、来源隔离、精确重复、IID manifest 全部通过。")
print("LOGO 将按各目标生成器从 Train 和 Validation 移除该生成器，并在以上完全相同的 Test 样本上评估。")

## 3. 定义与 IID 一致的训练和评估函数

这里不会调用 IID 已训练模型：每个 LOGO fold 都重新加载 `FacebookAI/roberta-base` 原始预训练权重，并随机初始化新的二分类头。最佳 epoch 由**不含目标生成器的 Validation** 选择。

In [ ]:
def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def score_binary(y, yhat, prob):
    y=np.asarray(y,dtype=int); yhat=np.asarray(yhat,dtype=int)
    r={
        "n":int(len(y)),
        "accuracy":float(accuracy_score(y,yhat)),
        "precision_ai":float(precision_score(y,yhat,zero_division=0)),
        "recall_ai":float(recall_score(y,yhat,zero_division=0)),
        "f1_ai":float(f1_score(y,yhat,zero_division=0)),
        "f1_macro":float(f1_score(y,yhat,average="macro",zero_division=0)),
        "confusion_matrix_human_ai":confusion_matrix(y,yhat,labels=[0,1]).tolist(),
    }
    r["roc_auc"] = float(roc_auc_score(y,prob)) if len(np.unique(y))==2 else None
    return r

class TokenizedData(Dataset):
    def __init__(self, frame, tokenizer):
        self.tokens=tokenizer(frame["text"].astype(str).tolist(), max_length=MAX_LENGTH, truncation=True, padding=False)
        self.labels=frame["label"].astype(int).tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {"input_ids":self.tokens["input_ids"][i],
                "attention_mask":self.tokens["attention_mask"][i], "labels":self.labels[i]}

def eval_model(model, loader, title):
    model.eval()
    ys, preds, probs, losses=[],[],[],[]
    with torch.inference_mode():
        for batch in tqdm(loader, desc=title, leave=False):
            batch={k:v.to(DEVICE) for k,v in batch.items()}
            with torch.autocast(device_type="cuda",dtype=torch.float16):
                result=model(**batch)
            p=torch.softmax(result.logits.float(), dim=-1)[:,1]
            ys.extend(batch["labels"].cpu().tolist())
            preds.extend(result.logits.argmax(-1).cpu().tolist())
            probs.extend(p.cpu().tolist())
            losses.append(float(result.loss.item()))
    metrics=score_binary(ys,preds,probs)
    metrics["mean_eval_loss"]=float(np.mean(losses))
    return metrics, preds, probs

def write_json(path, value):
    path.write_text(json.dumps(value,ensure_ascii=False,indent=2),encoding="utf-8")

print("[READY] LOGO 训练与评估函数定义成功。")

## 4. 定义单个 held-out 生成器的完整运行过程

每个 fold 将保存：验证阶段学习曲线、最佳 checkpoint、正式 Test 指标、逐条预测、配置和文件散列。

保护措施：已经完成的 fold 会被直接跳过；若发现上次中断留下的不完整 fold 文件夹，会停止并要求检查，**不会静默覆盖**。

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
assert tokenizer.model_max_length >= MAX_LENGTH, "MAX_LENGTH 超出模型范围"

DATA_SHA={fn:sha256_file(DATA_DIR/fn) for fn in SPLIT_FILES.values()}
IID_SHA={fn:sha256_file(IID_RESULTS/fn) for fn in ["iid_test_metrics.json","iid_per_generator_metrics.csv","per_generator_test_manifest.csv"]}


def run_logo_fold(target_gen):
    assert target_gen in ALL_GENERATORS
    fold_dir = LOGO_RESULTS / target_gen
    metrics_file = fold_dir / "logo_test_metrics.json"
    if metrics_file.exists():
        existing=json.loads(metrics_file.read_text(encoding="utf-8"))
        cfg=json.loads((fold_dir / "run_config.json").read_text(encoding="utf-8"))
        assert cfg["data_sha256"] == DATA_SHA and cfg["iid_sha256"] == IID_SHA, "数据或清单已变化，不能复用旧结果"
        assert cfg["training_params"] == TRAINING_PARAMS, "参数与历史 fold 不一致"
        print(f"[SKIP] {target_gen}: 已经完成，F1={existing['f1_ai']:.4f}")
        return existing
    if fold_dir.exists() and any(fold_dir.iterdir()):
        raise RuntimeError(f"发现未完成的历史文件夹 {fold_dir}。请先备份并检查，不会自动覆盖。")
    fold_dir.mkdir(parents=True, exist_ok=True)

    # The held-out generator must be ABSENT from both train and validation.
    train=splits["train"].loc[splits["train"]["model"] != target_gen].copy()
    val=splits["validation"].loc[splits["validation"]["model"] != target_gen].copy()
    if target_gen in set(train["model"]) | set(val["model"]):
        raise ValueError("held-out generator 泄漏进了 Train/Validation")
    assert set(train["label"]) == {0,1} and set(val["label"]) == {0,1}
    assert set(train.loc[train["label"]==1,"model"]) == set(ALL_GENERATORS)-{target_gen}
    assert set(val.loc[val["label"]==1,"model"]) == set(ALL_GENERATORS)-{target_gen}

    # Exact same test samples and order as recorded in IID manifest.
    part=manifest.loc[manifest["target_generator"]==target_gen].copy()
    test=by_id.loc[part["id"].tolist()].copy().reset_index(drop=True)
    assert len(test)==len(part) and list(test["id"])==list(part["id"])
    assert set(test.loc[test["label"]==1,"model"])=={target_gen}
    # Test selection uses only fixed manifest; no target generator text enters Train or Validation.
    fold_counts={"train_total":len(train),"train_human":int((train["label"]==0).sum()),
                 "train_ai":int((train["label"]==1).sum()),
                 "validation_total":len(val),"validation_human":int((val["label"]==0).sum()),
                 "validation_ai":int((val["label"]==1).sum()),
                 "test_total":len(test),"test_human":int((test["label"]==0).sum()),
                 "test_ai":int((test["label"]==1).sum())}
    print(f"\n===== LOGO: Hold out {target_gen} =====")
    print(fold_counts)
    print("训练 AI 生成器:", sorted(set(train.loc[train["label"]==1,"model"])))
    set_all_seeds(SEED)
    use_amp=True
    collator=DataCollatorWithPadding(tokenizer=tokenizer,padding=True,pad_to_multiple_of=8,return_tensors="pt")
    train_loader=DataLoader(TokenizedData(train, tokenizer),batch_size=BATCH_SIZE,shuffle=True,
                            generator=torch.Generator().manual_seed(SEED),num_workers=0,collate_fn=collator)
    val_loader=DataLoader(TokenizedData(val, tokenizer),batch_size=EVAL_BATCH_SIZE,shuffle=False,
                          num_workers=0,collate_fn=collator)
    model=AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,num_labels=2,id2label={0:"Human",1:"AI"},label2id={"Human":0,"AI":1}
    ).to(DEVICE)
    optim=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    steps_per_epoch=math.ceil(len(train_loader)/GRAD_ACCUM)
    sched=get_linear_schedule_with_warmup(optim, num_warmup_steps=int(steps_per_epoch*EPOCHS*WARMUP_RATIO),
                                         num_training_steps=steps_per_epoch*EPOCHS)
    scaler=torch.amp.GradScaler("cuda",enabled=use_amp)
    checkpoint=fold_dir/"best_model"
    best_f1=-1.; best_epoch=-1; stale=0; hist=[]
    try:
        for epoch in range(1,EPOCHS+1):
            model.train(); optim.zero_grad(set_to_none=True); loss_total=0.
            for step,batch in enumerate(tqdm(train_loader,desc=f"{target_gen}: epoch {epoch}/{EPOCHS}"),start=1):
                batch={k:v.to(DEVICE) for k,v in batch.items()}
                group_start=((step-1)//GRAD_ACCUM)*GRAD_ACCUM
                group_len=min(GRAD_ACCUM,len(train_loader)-group_start)
                with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=use_amp):
                    out=model(**batch)
                    loss=out.loss/group_len
                scaler.scale(loss).backward()
                loss_total += float(out.loss.item())
                if step % GRAD_ACCUM == 0 or step == len(train_loader):
                    scaler.unscale_(optim)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optim);scaler.update();sched.step();optim.zero_grad(set_to_none=True)
            vm,_,_=eval_model(model,val_loader,f"{target_gen} validation {epoch}")
            rec={"epoch":epoch,"mean_train_loss":loss_total/len(train_loader),
                 "validation_f1_ai":vm["f1_ai"],"validation_accuracy":vm["accuracy"],
                 "validation_f1_macro":vm["f1_macro"],"validation_recall_ai":vm["recall_ai"]}
            hist.append(rec)
            pd.DataFrame(hist).to_csv(fold_dir/"training_history.csv",index=False)
            print(f"{target_gen} epoch {epoch}: Validation F1={vm['f1_ai']:.4f}")
            if vm["f1_ai"]>best_f1+1e-8:
                best_f1=vm["f1_ai"];best_epoch=epoch;stale=0
                model.save_pretrained(checkpoint,safe_serialization=True)
                tokenizer.save_pretrained(checkpoint)
                print(f"[BEST] {target_gen}: epoch {best_epoch}")
            else:
                stale += 1
                if stale>=PATIENCE: break
    except torch.cuda.OutOfMemoryError as e:
        torch.cuda.empty_cache()
        raise RuntimeError("显存不足：先重启 Kernel，并在尚未完成任何 fold 时将 BATCH_SIZE=2、EVAL_BATCH_SIZE=2、GRAD_ACCUM=8。") from e

    if best_epoch<1: raise RuntimeError("没有保存可用 checkpoint")
    del model,optim,sched,scaler,train_loader,val_loader
    gc.collect();torch.cuda.empty_cache()
    model=AutoModelForSequenceClassification.from_pretrained(checkpoint).to(DEVICE)
    test_loader=DataLoader(TokenizedData(test,tokenizer),batch_size=EVAL_BATCH_SIZE,shuffle=False,
                           num_workers=0,collate_fn=collator)
    result,preds,probs=eval_model(model,test_loader,f"FINAL LOGO TEST: {target_gen}")
    result.update({"target_generator":target_gen,"best_epoch":int(best_epoch),
                   "best_validation_f1":float(best_f1),"n_human":fold_counts["test_human"],
                   "n_ai":fold_counts["test_ai"],"train_data_count":len(train),
                   "validation_data_count":len(val)})
    out_pred=part.copy()
    out_pred["prediction"]=preds
    out_pred["probability_ai"]=probs
    out_pred["correct"]=(out_pred["label"].astype(int).to_numpy()==np.asarray(preds))
    out_pred.to_csv(fold_dir/"logo_test_predictions.csv",index=False)
    write_json(fold_dir/"run_config.json",{
        "target_generator":target_gen,"model_name":MODEL_NAME,
        "training_params":TRAINING_PARAMS,"source_control":"original fixed IID grouped splits",
        "test_control":"identical original IID per-generator manifest IDs and order",
        "data_counts":fold_counts,"data_sha256":DATA_SHA,"iid_sha256":IID_SHA,
        "pytorch_version":torch.__version__,"transformers_version":transformers.__version__,
        "gpu":torch.cuda.get_device_name(0),"best_checkpoint_rule":"validation_f1_ai",
    })
    # Metrics written LAST: completed fold marker.
    write_json(metrics_file,result)
    del model,test_loader
    gc.collect();torch.cuda.empty_cache()
    print(f"[DONE] {target_gen} LOGO: AI F1={result['f1_ai']:.4f}; recall={result['recall_ai']:.4f}")
    return result

TRAINING_PARAMS={"max_length":MAX_LENGTH,"epochs":EPOCHS,"batch_size":BATCH_SIZE,
                 "eval_batch_size":EVAL_BATCH_SIZE,"gradient_accumulation_steps":GRAD_ACCUM,
                 "learning_rate":LEARNING_RATE,"weight_decay":WEIGHT_DECAY,
                 "warmup_ratio":WARMUP_RATIO,"patience":PATIENCE,"seed":SEED}
print("[READY] fold 训练函数已定义。下一个单元格会开始 GPU 训练。")

## 5. 正式 LOGO 训练（GPU 耗时单元格）

默认依次执行四个 fold。确认上方 `[PREFLIGHT PASSED]` 后再运行。每个 fold 完成时日志显示 `[DONE]` 和实际 F1。如只想先试跑一轮，先将 `TARGETS_TO_RUN` 设为 `["llama-chat"]`，试跑后改回 `ALL_GENERATORS`，从头运行到这里。完成且数据散列、参数一致的 fold 会被跳过。请勿在训练中途修改数据。


In [ ]:
LOGO_RESULTS.mkdir(parents=True,exist_ok=True)
for target in TARGETS_TO_RUN:
    _ = run_logo_fold(target)
print("[RUN COMPLETE] 当前请求的 LOGO folds 已处理完。")

## 6. 汇总相同测试样本上的 IID vs LOGO 成绩

只有完成的 fold 会出现在表中。四个生成器全部完成后才可据此撰写正式研究结论。**不要根据测试成绩返回调整参数**。

In [ ]:
rows=[]
for gen in ALL_GENERATORS:
    path=LOGO_RESULTS/gen/"logo_test_metrics.json"
    if not path.is_file(): continue
    logo=json.loads(path.read_text(encoding="utf-8"))
    iid=iid_by_gen.loc[iid_by_gen["target_generator"]==gen].iloc[0]
    assert int(iid["n"])==int(logo["n"]),"相同生成器的测试样本数量不同"
    rows.append({
        "generator":gen,"n_test":int(logo["n"]),
        "iid_accuracy":float(iid["accuracy"]),"logo_accuracy":float(logo["accuracy"]),
        "iid_f1_ai":float(iid["f1_ai"]),"logo_f1_ai":float(logo["f1_ai"]),
        "f1_change_logo_minus_iid":float(logo["f1_ai"])-float(iid["f1_ai"]),
        "iid_recall_ai":float(iid["recall_ai"]),"logo_recall_ai":float(logo["recall_ai"]),
        "logo_train_size":int(logo["train_data_count"]),
    })
comparison=pd.DataFrame(rows)
if len(comparison):
    out=LOGO_RESULTS/"iid_vs_logo_comparison.csv"
    comparison.to_csv(out,index=False)
    display(comparison.style.format({c:"{:.4f}" for c in comparison.columns if c not in ["generator","n_test","logo_train_size"]}))
    print("[SAVED]",out)
else:
    print("还没有完成 LOGO fold。")
print(f"已完成 {len(rows)}/4 个 held-out generators。")
if len(rows)<4:
    print("请先完成剩余的 LOGO fold，当前结果仅供流程检查，不能代表完整实验。")
else:
    print("[ALL DONE] 4 个 LOGO fold 已结束。请保存并上传 iid_vs_logo_comparison.csv、每个 fold 的 logo_test_metrics.json。")

## 7. 研究记录与下一步

请保存：

1. `logo_roberta_results_jupyter/iid_vs_logo_comparison.csv`（四组全部完成后）
2. `logo_roberta_results_jupyter/<生成器>/logo_test_metrics.json`（每个 fold 的指标）
3. 每个 fold 的 `run_config.json`、`training_history.csv` 和 `logo_test_predictions.csv`（复现与错误分析）

实验解释时必须注明：虽然测试文本完全一致，但 LOGO 训练集去掉了目标生成器，因此 AI 训练样本更少、训练集类别比例也不同；这是影响“泛化能力”解释的潜在混杂因素。本研究不能单凭 F1 下降证明 RoBERTa 是否真正“理解”AI 文本。若需要进一步严格归因，后续可考虑增加训练规模匹配的对照实验，但当前先完成约定的 IID / LOGO 两条件。

**不要删除或覆盖已有的 `iid_roberta_results_jupyter` 文件夹。**